# B2-024-gpu-scientific-ml-capstone — Practice p01 — Solution

**Type:** mc-normal-form · **Difficulty:** intro · **Concepts:** gpu-colab-l4-workflow

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Solution

Session 1's memory model counts, per parameter, a `float32` master weight, a `float32` gradient, and Adam's two `float32` moments, plus the activations kept for `backward()`.
Here $P=2{,}400{,}000$ and $E=12{,}000{,}000$.

| tensor group | elements | run 1 (all `float32`) | run 2 (`bfloat16` activations) |
|---|---:|---:|---:|
| master weights | $P$ | $4P=9{,}600{,}000$ B | $4P=9{,}600{,}000$ B |
| gradients | $P$ | $9{,}600{,}000$ B | $9{,}600{,}000$ B |
| Adam first moment $m$ | $P$ | $9{,}600{,}000$ B | $9{,}600{,}000$ B |
| Adam second moment $v$ | $P$ | $9{,}600{,}000$ B | $9{,}600{,}000$ B |
| activations | $E$ | $4E=48{,}000{,}000$ B | $2E=24{,}000{,}000$ B |
| **total** | | $86{,}400{,}000$ B | $62{,}400{,}000$ B |

Autocast changes only the activation term; the $16P=38{,}400{,}000$ bytes of `float32` state are identical in both runs.

**Ratio.** $\dfrac{62{,}400{,}000}{86{,}400{,}000}=\dfrac{624}{864}$.
Since $\gcd(624,864)=48$ ($624=48\cdot13$, $864=48\cdot18$), the normal form is $\dfrac{13}{18}$ with $\gcd(13,18)=1$, $b=18>0$.

So $a+b=13+18=31$.

**Answer: E.**

**Distractors.**

* **A (3)** is $1/2$: assuming mixed precision halves *all* memory (the pitfall "expecting mixed precision to halve memory").
* **B (5)** is $2/3$: also storing the master weights in `bfloat16` ($2P+4P+4P+4P+2E=57{,}600{,}000$ against $86{,}400{,}000$).
* **C (23)** is $9/14$: forgetting both Adam moments ($8P+2E$ over $8P+4E$, i.e. $43{,}200{,}000/67{,}200{,}000$).
* **D (27)** is $11/16$: counting only one Adam moment ($12P+2E$ over $12P+4E$, i.e. $52{,}800{,}000/76{,}800{,}000$).

The reversed ratio $18/13$ also sums to $31$, but it is the wrong quantity (run 1 over run 2, greater than 1) and gets no credit; the derivation above is run 2 over run 1.

In [ ]:
from fractions import Fraction
from math import gcd

P = 2_400_000
E = 12_000_000
FP32, BF16 = 4, 2

groups_run1 = {"weights": FP32 * P, "grads": FP32 * P, "adam_m": FP32 * P, "adam_v": FP32 * P, "activations": FP32 * E}
groups_run2 = {"weights": FP32 * P, "grads": FP32 * P, "adam_m": FP32 * P, "adam_v": FP32 * P, "activations": BF16 * E}
total1 = sum(groups_run1.values())
total2 = sum(groups_run2.values())
ratio = Fraction(total2, total1)          # Fraction reduces to lowest terms with a positive denominator
a, b = ratio.numerator, ratio.denominator


def ratio_sum(run2, run1):
    r = Fraction(run2, run1)
    return r.numerator + r.denominator


distractors = {
    "A": ratio_sum(1, 2),                                         # "everything halves"
    "B": ratio_sum(2 * P + 12 * P + 2 * E, 16 * P + 4 * E),        # master weights also bf16
    "C": ratio_sum(8 * P + 2 * E, 8 * P + 4 * E),                  # no Adam moments
    "D": ratio_sum(12 * P + 2 * E, 12 * P + 4 * E),                # one Adam moment
}
ANSWER = "E"
print(groups_run1, total1)
print(groups_run2, total2)
print("ratio", ratio, "a+b =", a + b, "distractors", distractors)

### Answer check

In [ ]:
assert total1 == 86_400_000 and total2 == 62_400_000
assert all(groups_run1[k] == groups_run2[k] == 9_600_000 for k in ("weights", "grads", "adam_m", "adam_v"))
assert groups_run1["activations"] == 48_000_000 and groups_run2["activations"] == 24_000_000
assert (a, b) == (13, 18) and gcd(a, b) == 1 and b > 0
assert a + b == 31
assert distractors == {"A": 3, "B": 5, "C": 23, "D": 27}
assert ANSWER == "E"